In [17]:
import pandas as pd
import numpy as np

In [18]:
import os
print(os.getcwd())

d:\INTERNSHIP\credit card fraud detection\notebooks


In [19]:
dataset = pd.read_csv('../data/processed/cleaned_fraud_data.csv')
dataset.head()

,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,...,year,weekend,night,age,age_group,log_amt,amount_group,city_pop_group,distance_km,distance_group
0,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,29209,33.9659,-80.9355,333497,...,2020,1,0,52,41-60,1.350667,Low,very large,24.561462,Nearby
1,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,84002,40.3207,-110.4360,302,...,2020,1,0,30,26-40,3.428813,Low,small,104.925092,Very Far
2,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,F,Bellmore,NY,11710,40.6729,-73.5365,34496,...,2020,1,0,50,41-60,3.744314,Low,large,59.080078,Far
3,fraud_Haley Group,misc_pos,60.05,M,Titusville,FL,32780,28.5697,-80.8191,54767,...,2020,1,0,33,26-40,4.111693,Medium,large,27.698567,Medium
4,fraud_Johnston-Casper,travel,3.19,M,Falmouth,MI,49632,44.2529,-85.0170,1126,...,2020,1,0,65,60+,1.432701,Low,medium,104.335106,Very Far


In [20]:
dataset.columns

Index(['merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'merch_lat', 'merch_long', 'is_fraud',
       'day_of_week', 'date', 'day', 'month', 'hour', 'minute', 'year',
       'weekend', 'night', 'age', 'age_group', 'log_amt', 'amount_group',
       'city_pop_group', 'distance_km', 'distance_group'],
      dtype='object')

In [21]:
drop_cols = [
    "merchant",
    "city",
    "job",
    "zip",
    "date",
    "day",
    "minute",
    "year"
]

dataset_model = dataset.drop(columns=drop_cols, errors="ignore")

unique merchant,city,job,date have unique value in 100s so dropping them 

In [22]:
# Drop columns for first model
drop_cols = ["merchant", "city", "job", "date", "zip", "day", "minute", "year"]

dataset_model = dataset.drop(columns=drop_cols, errors="ignore")

# Separate input and target
X = dataset_model.drop("is_fraud", axis=1)
y = dataset_model["is_fraud"]

# Identify categorical and numerical columns from X only
cat_model = X.select_dtypes(include=["object", "category"]).columns
num_model = X.select_dtypes(include=["int64", "float64"]).columns

# Check categorical columns
for i in cat_model:
    print("unique", i, X[i].nunique())

unique category 14
unique gender 2
unique state 50
unique age_group 4
unique amount_group 4
unique city_pop_group 4
unique distance_group 4


In [23]:
X.columns

Index(['category', 'amt', 'gender', 'state', 'lat', 'long', 'city_pop',
       'merch_lat', 'merch_long', 'day_of_week', 'month', 'hour', 'weekend',
       'night', 'age', 'age_group', 'log_amt', 'amount_group',
       'city_pop_group', 'distance_km', 'distance_group'],
      dtype='object')

In [24]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2,stratify=y)

In [25]:
print(X_train["age_group"].unique())
print(X_train["amount_group"].unique())
print(X_train["city_pop_group"].unique())
print(X_train["distance_group"].unique())

['60+' '26-40' '41-60' '0-25']
['Low' 'Medium' 'High' 'Very High']
['small' 'large' 'medium' 'very large']
['Very Far' 'Medium' 'Far' 'Nearby']


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Define the order for each ordinal feature
age_order = ['0-25', '26-40', '41-60', '60+']
amount_order = ['Low', 'Medium', 'High', 'Very High']
city_pop_order = ['small', 'medium', 'large', 'very large']

# First check this value in your notebook
print(X["distance_group"].unique())

distance_order = ['Nearby', 'Medium', 'Far', 'Very Far']
# Change this if your actual distance_group values are different

# Nominal columns = no natural order
nominal_cols = ['category', 'gender', 'state']

# Ordinal columns = natural order
ordinal_cols = ['age_group', 'amount_group', 'city_pop_group', 'distance_group']

# Numerical columns
num_model = X.select_dtypes(include=["int64", "float64"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_model),

        ('nominal', OneHotEncoder(handle_unknown='ignore',sparse_output=False), nominal_cols),

        ('ordinal', OrdinalEncoder(
            categories=[age_order, amount_order, city_pop_order, distance_order],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ), ordinal_cols)
    ],
    remainder='drop'
)

['Nearby' 'Very Far' 'Far' 'Medium']


In [27]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [28]:
import os
import pandas as pd

os.makedirs("../data/processed", exist_ok=True)


feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)

X_train_processed_df.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test_processed_df.to_csv("../data/processed/X_test_processed.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [ ]:
import os
import pickle

os.makedirs("../models", exist_ok=True)

with open("../models/preprocessor.pkl", "wb") as file:
    pickle.dump(preprocessor, file)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor, "../models/preprocessor_v2.pkl")
print("Saved with joblib ✅")

Saved with joblib ✅
